# 🔬 Your Research Project: From Question to Discovery

**Duration:** 120 minutes  
**Level:** Advanced

---

## Welcome, Researcher!

This is it - your moment to become a real biosignal researcher! In this notebook, you'll design and execute a complete research study from start to finish, just like scientists do in universities and research labs.

### Your Mission:

Design and conduct a research study to answer the question:

**"How does regular exercise affect heart rate variability in adults?"**

You'll:
- 📋 Design a research study
- 👥 Generate a synthetic cohort (group) of participants
- 📊 Collect and analyze biosignal data
- 📈 Perform statistical comparisons
- 📝 Create publication-quality visualizations
- ✍️ Write up your findings like a real research paper

This is the culmination of everything you've learned. Let's make a discovery! 🚀

## 📚 Step 1: Research Design

Before we code anything, we need to plan our study like real researchers do!

### Study Design:

**Research Question:**  
Does regular aerobic exercise improve heart rate variability in sedentary adults?

**Hypothesis:**  
Adults who engage in regular aerobic exercise will show higher HRV (SDNN and RMSSD) compared to sedentary adults.

**Study Type:**  
Cross-sectional comparison study

**Groups:**
- **Control Group:** Sedentary adults (< 1 hour exercise/week)
- **Exercise Group:** Active adults (≥ 4 hours exercise/week)

**Sample Size:**  
30 participants per group (60 total)

**Measurements:**
- 5-minute resting ECG recording
- HRV analysis (SDNN, RMSSD, frequency domain metrics)
- Demographic data (age, BMI)

**Statistical Analysis:**
- Independent t-tests for group comparisons
- Effect size calculation (Cohen's d)
- Correlation analysis

Let's execute this study!

## 📦 Step 2: Loading Research Tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from scipy.interpolate import interp1d
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for publication-quality figures
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

print("✅ Research tools loaded!")
print("   Ready to conduct professional biosignal research!")

## 🧬 Step 3: Generating the Cohort

Let's create our synthetic study population with realistic characteristics!

In [ ]:
# Import HRV functions from previous notebook
def generate_ecg_signal(duration=300, sampling_rate=250, base_hr=70, hrv_std=50):
    """Generate realistic ECG with HRV."""
    t = np.linspace(0, duration, duration * sampling_rate)
    n_samples = len(t)
    
    mean_rr = 60.0 / base_hr
    rr_intervals = []
    current_time = 0
    
    while current_time < duration:
        respiratory_modulation = 0.1 * np.sin(2 * np.pi * 0.25 * current_time)
        hrv_noise = np.random.normal(0, hrv_std / 1000)
        rr = mean_rr * (1 + respiratory_modulation + hrv_noise)
        rr = max(0.3, min(2.0, rr))
        rr_intervals.append(rr)
        current_time += rr
    
    r_peak_times = np.cumsum(rr_intervals[:-1])
    ecg = np.zeros(n_samples)
    
    for peak_time in r_peak_times:
        if peak_time >= duration:
            break
        peak_idx = int(peak_time * sampling_rate)
        qrs_start = max(0, peak_idx - int(0.04 * sampling_rate))
        qrs_end = min(n_samples, peak_idx + int(0.04 * sampling_rate))
        qrs_width = qrs_end - qrs_start
        if qrs_width > 0:
            r_peak = signal.windows.gaussian(qrs_width, std=qrs_width/8)
            ecg[qrs_start:qrs_end] += r_peak * 1.5
    
    baseline = 0.05 * np.sin(2 * np.pi * 0.1 * t)
    noise = np.random.normal(0, 0.02, n_samples)
    ecg = ecg + baseline + noise
    
    return t, ecg, r_peak_times, np.array(rr_intervals[:-1]) * 1000

def calculate_hrv_metrics(rr_intervals):
    """Calculate comprehensive HRV metrics."""
    results = {}
    
    # Time domain
    results['mean_rr'] = np.mean(rr_intervals)
    results['mean_hr'] = 60000 / results['mean_rr']
    results['sdnn'] = np.std(rr_intervals, ddof=1)
    
    diff_rr = np.diff(rr_intervals)
    results['rmssd'] = np.sqrt(np.mean(diff_rr**2))
    results['pnn50'] = 100 * np.sum(np.abs(diff_rr) > 50) / len(diff_rr)
    
    # Frequency domain
    rr_times = np.cumsum(rr_intervals) / 1000
    rr_times = np.insert(rr_times, 0, 0)
    rr_values = np.insert(rr_intervals, 0, rr_intervals[0])
    
    f = interp1d(rr_times, rr_values, kind='cubic', fill_value='extrapolate')
    t_resampled = np.arange(0, rr_times[-1], 0.25)
    rr_resampled = f(t_resampled)
    
    freqs, psd = signal.welch(rr_resampled, fs=4, nperseg=256)
    
    lf_power = np.trapz(psd[(freqs >= 0.04) & (freqs < 0.15)],
                       freqs[(freqs >= 0.04) & (freqs < 0.15)])
    hf_power = np.trapz(psd[(freqs >= 0.15) & (freqs < 0.4)],
                       freqs[(freqs >= 0.15) & (freqs < 0.4)])
    
    results['lf_power'] = lf_power
    results['hf_power'] = hf_power
    results['lf_hf_ratio'] = lf_power / hf_power if hf_power > 0 else 0
    
    return results

print("🔧 Analysis functions ready!")

In [ ]:
# Generate cohort demographics
print("👥 Generating study cohort...\n")

np.random.seed(42)  # For reproducibility

n_per_group = 30

# SEDENTARY GROUP
sedentary_participants = []
for i in range(n_per_group):
    age = np.random.randint(25, 55)
    bmi = np.random.normal(27, 4)  # Slightly higher BMI
    bmi = max(20, min(35, bmi))  # Realistic range
    
    # Sedentary individuals typically have:
    # - Higher resting HR
    # - Lower HRV
    base_hr = np.random.normal(75, 8)
    hrv_std = np.random.normal(40, 10)  # Lower HRV
    
    sedentary_participants.append({
        'id': f'SED_{i+1:03d}',
        'group': 'Sedentary',
        'age': age,
        'bmi': bmi,
        'base_hr': base_hr,
        'hrv_std': hrv_std,
        'exercise_hours_per_week': np.random.uniform(0, 1)
    })

# EXERCISE GROUP
exercise_participants = []
for i in range(n_per_group):
    age = np.random.randint(25, 55)
    bmi = np.random.normal(24, 3)  # Lower BMI
    bmi = max(19, min(30, bmi))
    
    # Active individuals typically have:
    # - Lower resting HR
    # - Higher HRV
    base_hr = np.random.normal(62, 7)
    hrv_std = np.random.normal(65, 12)  # Higher HRV
    
    exercise_participants.append({
        'id': f'EX_{i+1:03d}',
        'group': 'Exercise',
        'age': age,
        'bmi': bmi,
        'base_hr': base_hr,
        'hrv_std': hrv_std,
        'exercise_hours_per_week': np.random.uniform(4, 8)
    })

# Combine into DataFrame
all_participants = sedentary_participants + exercise_participants
df = pd.DataFrame(all_participants)

print("✅ Cohort generated!\n")
print("COHORT SUMMARY:")
print("="*70)
print(df.groupby('group')[['age', 'bmi', 'exercise_hours_per_week']].describe().round(2))
print("\n📊 Total participants: 60 (30 per group)")

## 🔬 Step 4: Data Collection

Now let's "collect" ECG data from all participants!

In [ ]:
print("📡 Collecting ECG data from all participants...\n")
print("This will take a moment - simulating 60 five-minute recordings!\n")

# Collect data from all participants
for idx, participant in enumerate(all_participants):
    # Generate ECG
    _, _, _, rr_intervals = generate_ecg_signal(
        duration=300,
        base_hr=participant['base_hr'],
        hrv_std=participant['hrv_std']
    )
    
    # Calculate HRV metrics
    hrv_metrics = calculate_hrv_metrics(rr_intervals)
    
    # Store results
    participant.update(hrv_metrics)
    
    # Progress indicator
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1}/60 participants...")

# Update DataFrame
df = pd.DataFrame(all_participants)

print("\n✅ Data collection complete!\n")
print("📊 Collected HRV data from all 60 participants")
print(f"   Variables measured: {len(df.columns)}")
print(f"   HRV metrics calculated per participant: SDNN, RMSSD, pNN50, LF, HF, LF/HF")

Let's preview our research dataset!

In [ ]:
# Display first few participants from each group
print("📋 RESEARCH DATASET PREVIEW\n")
print("Sedentary Group (first 5):")
print(df[df['group'] == 'Sedentary'][['id', 'age', 'bmi', 'mean_hr', 'sdnn', 'rmssd']].head())
print("\nExercise Group (first 5):")
print(df[df['group'] == 'Exercise'][['id', 'age', 'bmi', 'mean_hr', 'sdnn', 'rmssd']].head())

# Save to CSV for future use
# df.to_csv('research_study_data.csv', index=False)
print("\n💾 Dataset ready for analysis!")

## 📊 Step 5: Descriptive Statistics

Let's describe our data - this is the first step in any research paper!

In [ ]:
# Calculate descriptive statistics by group
print("📊 DESCRIPTIVE STATISTICS\n")
print("="*80)

variables_of_interest = ['age', 'bmi', 'mean_hr', 'sdnn', 'rmssd', 'pnn50', 
                         'lf_power', 'hf_power', 'lf_hf_ratio']

for group in ['Sedentary', 'Exercise']:
    print(f"\n{group.upper()} GROUP (n={len(df[df['group']==group])})")
    print("-" * 80)
    
    group_data = df[df['group'] == group]
    
    for var in variables_of_interest:
        mean = group_data[var].mean()
        std = group_data[var].std()
        print(f"{var:20s}: {mean:8.2f} ± {std:6.2f}")

print("\n" + "="*80)

## 📈 Step 6: Data Visualization

Create publication-quality figures!

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('HRV Comparison: Sedentary vs. Exercise Groups', 
             fontsize=16, fontweight='bold', y=1.00)

# Define colors
colors = {'Sedentary': '#e74c3c', 'Exercise': '#3498db'}

# Plot 1: SDNN comparison
for group in ['Sedentary', 'Exercise']:
    data = df[df['group'] == group]['sdnn']
    axes[0, 0].hist(data, alpha=0.6, label=group, bins=15, color=colors[group])
axes[0, 0].set_xlabel('SDNN (ms)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('A) SDNN Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: RMSSD comparison
for group in ['Sedentary', 'Exercise']:
    data = df[df['group'] == group]['rmssd']
    axes[0, 1].hist(data, alpha=0.6, label=group, bins=15, color=colors[group])
axes[0, 1].set_xlabel('RMSSD (ms)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('B) RMSSD Distribution', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Heart Rate comparison
for group in ['Sedentary', 'Exercise']:
    data = df[df['group'] == group]['mean_hr']
    axes[0, 2].hist(data, alpha=0.6, label=group, bins=15, color=colors[group])
axes[0, 2].set_xlabel('Heart Rate (bpm)', fontsize=11)
axes[0, 2].set_ylabel('Frequency', fontsize=11)
axes[0, 2].set_title('C) Resting Heart Rate', fontsize=12, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(axis='y', alpha=0.3)

# Plot 4: Box plots for SDNN
bp1 = axes[1, 0].boxplot([df[df['group']=='Sedentary']['sdnn'],
                          df[df['group']=='Exercise']['sdnn']],
                         labels=['Sedentary', 'Exercise'],
                         patch_artist=True)
for patch, color in zip(bp1['boxes'], [colors['Sedentary'], colors['Exercise']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1, 0].set_ylabel('SDNN (ms)', fontsize=11)
axes[1, 0].set_title('D) SDNN Box Plot', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 5: Box plots for RMSSD
bp2 = axes[1, 1].boxplot([df[df['group']=='Sedentary']['rmssd'],
                          df[df['group']=='Exercise']['rmssd']],
                         labels=['Sedentary', 'Exercise'],
                         patch_artist=True)
for patch, color in zip(bp2['boxes'], [colors['Sedentary'], colors['Exercise']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1, 1].set_ylabel('RMSSD (ms)', fontsize=11)
axes[1, 1].set_title('E) RMSSD Box Plot', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

# Plot 6: Scatter plot - Age vs SDNN
for group in ['Sedentary', 'Exercise']:
    group_data = df[df['group'] == group]
    axes[1, 2].scatter(group_data['age'], group_data['sdnn'], 
                      alpha=0.6, s=80, label=group, color=colors[group])
axes[1, 2].set_xlabel('Age (years)', fontsize=11)
axes[1, 2].set_ylabel('SDNN (ms)', fontsize=11)
axes[1, 2].set_title('F) Age vs. SDNN', fontsize=12, fontweight='bold')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Publication-quality figures generated!")

## 🧮 Step 7: Statistical Analysis

Time to test our hypothesis using statistics!

In [ ]:
def cohens_d(group1, group2):
    """
    Calculate Cohen's d effect size.
    d = 0.2 (small), 0.5 (medium), 0.8 (large)
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std

def interpret_effect_size(d):
    """Interpret Cohen's d."""
    abs_d = abs(d)
    if abs_d < 0.2:
        return "negligible"
    elif abs_d < 0.5:
        return "small"
    elif abs_d < 0.8:
        return "medium"
    else:
        return "large"

print("🧮 STATISTICAL ANALYSIS\n")
print("="*80)
print("\nINDEPENDENT T-TESTS: Comparing Sedentary vs. Exercise Groups")
print("-" * 80)
print(f"{'Variable':<20} {'Sed Mean':>10} {'Ex Mean':>10} {'t-stat':>10} "
      f"{'p-value':>10} {'Cohen d':>10} {'Effect':>12}")
print("-" * 80)

# Get group data
sedentary = df[df['group'] == 'Sedentary']
exercise = df[df['group'] == 'Exercise']

# Perform t-tests for each variable
results = []
for var in ['mean_hr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf_ratio']:
    sed_data = sedentary[var]
    ex_data = exercise[var]
    
    # Perform independent t-test
    t_stat, p_value = stats.ttest_ind(sed_data, ex_data)
    
    # Calculate effect size
    d = cohens_d(ex_data, sed_data)  # Exercise - Sedentary
    effect = interpret_effect_size(d)
    
    # Store results
    results.append({
        'variable': var,
        'sed_mean': sed_data.mean(),
        'ex_mean': ex_data.mean(),
        't_stat': t_stat,
        'p_value': p_value,
        'cohens_d': d,
        'effect': effect,
        'significant': p_value < 0.05
    })
    
    # Print results
    sig_marker = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"{var:<20} {sed_data.mean():>10.2f} {ex_data.mean():>10.2f} "
          f"{t_stat:>10.3f} {p_value:>10.4f} {d:>10.3f} {effect:>12} {sig_marker}")

print("-" * 80)
print("Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print("Effect sizes: negligible (<0.2), small (0.2-0.5), medium (0.5-0.8), large (>0.8)")

# Store results
results_df = pd.DataFrame(results)

## 🔍 Step 8: Correlation Analysis

Let's explore relationships between variables!

In [ ]:
# Calculate correlations
print("\n🔍 CORRELATION ANALYSIS\n")
print("="*80)

# Variables to correlate
corr_vars = ['age', 'bmi', 'exercise_hours_per_week', 'mean_hr', 'sdnn', 'rmssd']
corr_matrix = df[corr_vars].corr()

# Create correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix: Study Variables', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Print notable correlations
print("\nKEY CORRELATIONS:\n")
print(f"Exercise hours <-> SDNN:  r = {df['exercise_hours_per_week'].corr(df['sdnn']):.3f}")
print(f"Exercise hours <-> RMSSD: r = {df['exercise_hours_per_week'].corr(df['rmssd']):.3f}")
print(f"Exercise hours <-> HR:    r = {df['exercise_hours_per_week'].corr(df['mean_hr']):.3f}")
print(f"Age <-> SDNN:             r = {df['age'].corr(df['sdnn']):.3f}")
print(f"BMI <-> SDNN:             r = {df['bmi'].corr(df['sdnn']):.3f}")

print("\n💡 Interpretation:")
print("   - Positive r values: variables increase together")
print("   - Negative r values: one increases as other decreases")
print("   - |r| > 0.3: moderate correlation, |r| > 0.5: strong correlation")

## 📊 Step 9: Effect Size Visualization

In [ ]:
# Visualize effect sizes
fig, ax = plt.subplots(figsize=(12, 8))

# Prepare data
variables = results_df['variable'].tolist()
effect_sizes = results_df['cohens_d'].tolist()
p_values = results_df['p_value'].tolist()

# Color by significance
colors_list = ['#2ecc71' if p < 0.001 else '#3498db' if p < 0.01 
               else '#f39c12' if p < 0.05 else '#95a5a6' 
               for p in p_values]

# Create bar plot
bars = ax.barh(variables, effect_sizes, color=colors_list, alpha=0.7, edgecolor='black')

# Add reference lines
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.axvline(x=0.2, color='gray', linestyle='--', alpha=0.5, label='Small effect')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Medium effect')
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5, label='Large effect')
ax.axvline(x=-0.2, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=-0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=-0.8, color='gray', linestyle='--', alpha=0.5)

# Add value labels
for i, (var, d, p) in enumerate(zip(variables, effect_sizes, p_values)):
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.text(d + 0.05 if d > 0 else d - 0.05, i, f'd={d:.2f} {sig}', 
            va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Effect Size (Cohen\'s d)\n← Favors Sedentary | Favors Exercise →', 
              fontsize=12, fontweight='bold')
ax.set_title('Effect Sizes: Exercise Group vs. Sedentary Group', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Create custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', alpha=0.7, label='p < 0.001 ***'),
    Patch(facecolor='#3498db', alpha=0.7, label='p < 0.01 **'),
    Patch(facecolor='#f39c12', alpha=0.7, label='p < 0.05 *'),
    Patch(facecolor='#95a5a6', alpha=0.7, label='Not significant')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📊 Effect size plot generated!")

## ✍️ Step 10: Writing Up the Findings

Let's create a research abstract and results summary!

In [ ]:
def generate_research_abstract(results_df, df):
    """
    Generate a research paper style abstract.
    """
    # Get key statistics
    sdnn_result = results_df[results_df['variable'] == 'sdnn'].iloc[0]
    rmssd_result = results_df[results_df['variable'] == 'rmssd'].iloc[0]
    hr_result = results_df[results_df['variable'] == 'mean_hr'].iloc[0]
    
    abstract = f"""
{'='*80}
                              RESEARCH ABSTRACT
{'='*80}

Title: The Effects of Regular Aerobic Exercise on Heart Rate Variability 
       in Adults: A Cross-Sectional Study

Authors: [Your Name], Research Team
Date: {datetime.now().strftime('%B %d, %Y')}

{'='*80}
ABSTRACT
{'='*80}

Background:
Heart rate variability (HRV) is a non-invasive measure of autonomic nervous
system function and cardiovascular health. Regular aerobic exercise has been
hypothesized to improve HRV, but the magnitude of this effect in community-
dwelling adults requires further investigation.

Objective:
To compare HRV metrics between sedentary adults and those engaging in regular
aerobic exercise, and to quantify the effect size of exercise on cardiac
autonomic function.

Methods:
This cross-sectional study recruited 60 adults (age range: 25-55 years) divided
into two groups: sedentary (n=30, <1 hour exercise/week) and exercise (n=30,
≥4 hours exercise/week). Five-minute resting ECG recordings were obtained from
all participants. HRV was assessed using standard time-domain (SDNN, RMSSD,
pNN50) and frequency-domain (LF, HF, LF/HF ratio) metrics. Group comparisons
were performed using independent t-tests, and effect sizes were calculated
using Cohen's d.

Results:
The exercise group demonstrated significantly higher SDNN compared to the
sedentary group ({sdnn_result['ex_mean']:.1f}±{exercise['sdnn'].std():.1f} ms vs. 
{sdnn_result['sed_mean']:.1f}±{sedentary['sdnn'].std():.1f} ms, p={sdnn_result['p_value']:.4f}, 
d={sdnn_result['cohens_d']:.2f}). Similarly, RMSSD was significantly elevated in the 
exercise group ({rmssd_result['ex_mean']:.1f}±{exercise['rmssd'].std():.1f} ms vs. 
{rmssd_result['sed_mean']:.1f}±{sedentary['rmssd'].std():.1f} ms, p={rmssd_result['p_value']:.4f}, 
d={rmssd_result['cohens_d']:.2f}). Resting heart rate was significantly lower in the 
exercise group ({hr_result['ex_mean']:.1f}±{exercise['mean_hr'].std():.1f} bpm vs. 
{hr_result['sed_mean']:.1f}±{sedentary['mean_hr'].std():.1f} bpm, p={hr_result['p_value']:.4f}, 
d={hr_result['cohens_d']:.2f}).

Conclusion:
Regular aerobic exercise is associated with significantly improved HRV and
reduced resting heart rate in community-dwelling adults. The observed effect
sizes suggest clinically meaningful improvements in cardiac autonomic function.
These findings support the recommendation of regular aerobic exercise for
cardiovascular health optimization.

Keywords: heart rate variability, exercise, autonomic nervous system, 
          cardiovascular health, SDNN, RMSSD

{'='*80}
    """
    
    return abstract

# Generate and print abstract
abstract = generate_research_abstract(results_df, df)
print(abstract)

## 📝 Step 11: Research Summary Table

In [ ]:
# Create publication-ready results table
print("\n📋 TABLE 1: Group Comparisons and Statistical Results\n")
print("="*95)
print(f"{'Variable':<15} {'Sedentary (n=30)':<20} {'Exercise (n=30)':<20} "
      f"{'p-value':<12} {'Effect Size':<15} {'Interp.'}")
print("="*95)

for _, row in results_df.iterrows():
    var = row['variable']
    
    # Get standard deviations
    sed_std = sedentary[var].std()
    ex_std = exercise[var].std()
    
    # Format output
    sed_str = f"{row['sed_mean']:.2f} ± {sed_std:.2f}"
    ex_str = f"{row['ex_mean']:.2f} ± {ex_std:.2f}"
    p_str = f"{row['p_value']:.4f}"
    d_str = f"d = {row['cohens_d']:.2f}"
    
    sig_marker = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else "ns"
    
    print(f"{var:<15} {sed_str:<20} {ex_str:<20} {p_str:<12} {d_str:<15} {row['effect']:>10} {sig_marker}")

print("="*95)
print("Values are Mean ± SD")
print("Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print("Effect sizes: Cohen's d interpreted as small (0.2), medium (0.5), or large (0.8)")
print("\nAbbreviations:")
print("  SDNN: Standard deviation of NN intervals")
print("  RMSSD: Root mean square of successive differences")
print("  pNN50: Percentage of successive NN differences > 50ms")
print("  LF: Low frequency power, HF: High frequency power")

## 🎯 Step 12: Key Findings Summary

In [ ]:
print("\n🎯 KEY FINDINGS SUMMARY\n")
print("="*80)

findings = []

# SDNN finding
sdnn_result = results_df[results_df['variable'] == 'sdnn'].iloc[0]
if sdnn_result['significant']:
    percent_diff = ((sdnn_result['ex_mean'] - sdnn_result['sed_mean']) / sdnn_result['sed_mean']) * 100
    findings.append(f"""1. SDNN (Primary HRV Metric):
   - Exercise group showed {percent_diff:.1f}% higher SDNN than sedentary group
   - This difference was statistically significant (p={sdnn_result['p_value']:.4f})
   - Effect size: {sdnn_result['effect']} (d={sdnn_result['cohens_d']:.2f})
   - Clinical significance: Indicates better cardiac autonomic function""")

# RMSSD finding
rmssd_result = results_df[results_df['variable'] == 'rmssd'].iloc[0]
if rmssd_result['significant']:
    percent_diff = ((rmssd_result['ex_mean'] - rmssd_result['sed_mean']) / rmssd_result['sed_mean']) * 100
    findings.append(f"""\n2. RMSSD (Parasympathetic Activity):
   - Exercise group showed {percent_diff:.1f}% higher RMSSD than sedentary group
   - This difference was statistically significant (p={rmssd_result['p_value']:.4f})
   - Effect size: {rmssd_result['effect']} (d={rmssd_result['cohens_d']:.2f})
   - Clinical significance: Suggests enhanced parasympathetic tone""")

# Heart rate finding
hr_result = results_df[results_df['variable'] == 'mean_hr'].iloc[0]
if hr_result['significant']:
    bpm_diff = hr_result['sed_mean'] - hr_result['ex_mean']
    findings.append(f"""\n3. Resting Heart Rate:
   - Exercise group had {bpm_diff:.1f} bpm lower resting heart rate
   - This difference was statistically significant (p={hr_result['p_value']:.4f})
   - Effect size: {hr_result['effect']} (d={abs(hr_result['cohens_d']):.2f})
   - Clinical significance: Lower resting HR indicates better fitness""")

# Print findings
for finding in findings:
    print(finding)

print("\n" + "="*80)
print("\n💡 OVERALL INTERPRETATION:\n")
print("This study provides strong evidence that regular aerobic exercise is")
print("associated with improved heart rate variability and cardiac autonomic")
print("function in adults. The observed improvements in SDNN, RMSSD, and resting")
print("heart rate all indicate enhanced cardiovascular health and stress resilience.")
print("\nThe large effect sizes (Cohen's d > 0.8 for key metrics) suggest these")
print("differences are not only statistically significant but also clinically")
print("meaningful, supporting the public health recommendation for regular physical")
print("activity.")
print("\n" + "="*80)

## 🎉 Congratulations - You're a Biosignal Researcher!

### 🏆 What You Just Accomplished:

You just completed a **COMPLETE RESEARCH STUDY** from beginning to end! This is exactly what researchers do in universities, hospitals, and research institutions. Look at what you did:

#### ✅ Research Skills Mastered:

1. **Study Design**: Formulated research question and hypothesis
2. **Cohort Generation**: Created realistic study population
3. **Data Collection**: Simulated biosignal recordings from 60 participants
4. **Descriptive Statistics**: Characterized your study population
5. **Data Visualization**: Created publication-quality figures
6. **Statistical Testing**: Performed t-tests and calculated effect sizes
7. **Correlation Analysis**: Explored relationships between variables
8. **Results Interpretation**: Drew meaningful conclusions
9. **Scientific Writing**: Wrote a research abstract
10. **Results Communication**: Created professional tables and summaries

### 📊 Your Research Findings:

Your study found that regular exercise is associated with:
- **Significantly higher HRV** (both SDNN and RMSSD)
- **Lower resting heart rate**
- **Large effect sizes** indicating clinically meaningful differences
- **Strong statistical evidence** (p < 0.001 for key metrics)

### 🌟 This Is Real Science!

The methods you used are:
- The **same statistical approaches** used in published research
- The **same visualization techniques** used in scientific journals
- The **same interpretation framework** used by researchers worldwide

Your synthetic data simulated real-world patterns, but the analysis techniques are 100% authentic!

### 🚀 What's Next?

Now that you can design and execute research studies, you could:

1. **Design your own studies**:
   - Compare meditation vs. exercise effects on HRV
   - Investigate age-related changes in cardiac function
   - Explore effects of sleep quality on autonomic function

2. **Work with real data**:
   - Apply these techniques to actual biosensor recordings
   - Collaborate with researchers or clinicians
   - Contribute to open-source biosignal datasets

3. **Advanced topics**:
   - Machine learning for biosignal classification
   - Real-time monitoring systems
   - Longitudinal study designs
   - Multi-modal biosignal integration

### 💭 Reflection Questions:

- What surprised you most about the research process?
- How would you improve this study design?
- What other research questions interest you?
- How could you apply these skills to real-world problems?

### 🎓 Final Thoughts:

You started this learning journey learning about signals and noise. Now you can:
- Generate and analyze complex biosignals
- Build AI systems for health monitoring
- Perform clinical-grade HRV analysis
- Design and execute complete research studies

**That's not just learning - that's transformation!**

You now have the skills to contribute to:
- Healthcare technology development
- Clinical research
- Public health initiatives
- Preventive medicine
- Wearable technology innovation

---

## 🌟 You Are Now a Biosignal Researcher!

Remember: Every expert was once a beginner who never gave up. You've proven you have what it takes.

**Keep exploring, keep questioning, keep discovering!**

The world needs more people who can bridge technology and health. You're one of them now. 🚀

---

*End of Learning Series*

## 🎯 Extension Exercises (Optional)

Want to keep going? Try these challenges:

### Challenge 1: Add a Third Group
Add a "moderate exercise" group (2-3 hours/week) and see if there's a dose-response relationship!

### Challenge 2: Longitudinal Study
Design a before/after study tracking sedentary individuals who start an exercise program.

### Challenge 3: Confounding Variables
Investigate how age or BMI might confound the exercise-HRV relationship using statistical controls.

### Challenge 4: Power Analysis
Calculate the minimum sample size needed to detect these effects with 80% power.

### Challenge 5: Publication
Write a complete "Methods" and "Discussion" section for this study!

### Challenge 6: Different Research Question
Design a completely new study investigating a different aspect of biosignals!

In [ ]:
# YOUR EXTENDED RESEARCH HERE!
# Use this space to try the challenges above or design your own study

print("🚀 Ready for your next research adventure!")